# 🎵 Raga Bhairav: Unconditioned Melody Generation with LSTM

**Task 1 — Symbolic, Unconditioned Generation**

We train an LSTM to learn the melodic distribution of **Raga Bhairav**, one of the oldest and most revered ragas in Indian classical music. The model learns from synthetically generated sequences encoding Bhairav's grammar, then generates new melodies by sampling from the learned distribution. The final output is a full 3-track performance: LSTM melody (Sitar) + Tanpura drone + Tabla rhythm (Teentaal).

### What is Raga Bhairav?
- **Time of performance**: Dawn
- **Mood (rasa)**: Serious, devotional, profound
- **Arohana** (ascent):  S r G m P d N S'
- **Avarohana** (descent): S' N d P m G r S
- **Vadi** (most important note): Ma (F)
- **Samvadi** (second most important): Sa (C)
- **Key feature**: Flat 2nd (komal Re) and flat 6th (komal Dha) give it a uniquely ancient, devotional sound

### Pipeline
1. Encode Bhairav grammar → generate 5000 synthetic training sequences
2. Tokenize as `(pitch, duration)` pairs
3. Train 2-layer LSTM (next-token prediction)
4. Sample new melodies at multiple temperatures
5. Assemble full performance: Sitar + Tanpura drone + Tabla (Teentaal)
6. Export MIDI → MP3

## 0. Install Dependencies

In [ ]:
!pip install music21 -q
!apt-get install -y fluidsynth fluid-soundfont-gm ffmpeg -q
!pip install pyfluidsynth -q
print('Done.')

## 1. Imports & Setup

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from music21 import stream, note, instrument, tempo
from collections import Counter
import matplotlib.pyplot as plt
import subprocess
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 2. Raga Bhairav Grammar

We encode Bhairav's rules explicitly rather than relying on existing recordings:
- **Scale**: Sa (C), komal Re (Db), Ga (E), Ma (F), Pa (G), komal Dha (Ab), Ni (B)
- **Arohana / Avarohana**: separate ascending and descending orderings
- **Vadi / Samvadi**: Ma and Sa are weighted more heavily during generation
- **Pakad phrases**: characteristic seed sequences that define Bhairav's identity
- **Gamakas**: ornamental oscillations around key notes, common in Indian classical music

In [ ]:
BHAIRAV = {
    'name': 'Bhairav',
    # Arohana: ascending — S r G m P d N S'
    'arohana':  [60, 61, 64, 65, 67, 68, 71, 72],
    # Avarohana: descending — S' N d P m G r S
    'avarohana': [72, 71, 68, 67, 65, 64, 61, 60],
    # Full note set across 3 octaves for generation
    'notes': [
        48, 49, 52, 53, 55, 56, 59,   # octave 3 (low)
        60, 61, 64, 65, 67, 68, 71,   # octave 4 (middle)
        72, 73, 76, 77, 79, 80, 83,   # octave 5 (high)
    ],
    # Vadi (Ma) and Samvadi (Sa) across octaves
    'vadi':    [53, 65, 77],
    'samvadi': [48, 60, 72],
    # Pakad: characteristic phrases that define Bhairav's identity
    'pakad': [
        [61, 60, 68, 67, 65],              # Re Sa Dha Pa Ma — classic opening
        [64, 65, 67, 68, 67, 65],          # Ga Ma Pa Dha Pa Ma
        [71, 72, 71, 68, 67],              # Ni Sa' Ni Dha Pa
        [60, 61, 64, 65, 64, 61, 60],      # Sa Re Ga Ma Ga Re Sa
        [65, 67, 68, 71, 72],              # Ma Pa Dha Ni Sa'
        [72, 71, 68, 67, 65, 64, 61, 60],  # full avarohana
    ],
    # Gamakas: ornamental turns around key notes
    'gamaka': [
        [65, 64, 65],   # Ma Ga Ma
        [67, 68, 67],   # Pa Dha Pa
        [61, 60, 61],   # Re Sa Re
        [71, 72, 71],   # Ni Sa' Ni
    ]
}

# Duration vocabulary (quarter note units) and sampling weights
DURATIONS    = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
DUR_WEIGHTS  = [0.15, 0.35, 0.10, 0.25, 0.10, 0.05]

print(f"Raga: {BHAIRAV['name']}")
print(f"Scale: {len(set(BHAIRAV['notes']))} unique pitches across 3 octaves")
print(f"Pakad phrases: {len(BHAIRAV['pakad'])}")
print(f"Duration vocab: {DURATIONS}")

## 3. Synthetic Data Generation

We generate training sequences using a **grammar-constrained random walk**:
- Notes are weighted by stepwise motion preference, current direction, and vadi/samvadi importance
- Direction flips every 4–10 steps to create natural melodic arcs
- Pakad phrases and gamakas are injected stochastically for authenticity
- Each token is a `(pitch_midi, duration)` pair

In [ ]:
def get_note_weights(current_pitch, raga, direction):
    """Compute sampling weights for next note based on raga grammar rules."""
    weights = []
    for p in raga['notes']:
        w = 1.0
        interval = p - current_pitch
        # Prefer stepwise motion
        if   abs(interval) <= 2: w *= 3.0
        elif abs(interval) <= 4: w *= 1.5
        else:                    w *= 0.4
        # Prefer notes in current direction
        if direction == 'up'   and interval > 0: w *= 1.8
        if direction == 'down' and interval < 0: w *= 1.8
        # Boost vadi and samvadi
        if p in raga['vadi']:    w *= 2.5
        if p in raga['samvadi']: w *= 1.8
        # Discourage repeating same note
        if p == current_pitch:   w *= 0.3
        weights.append(w)
    return weights


def generate_sequence(raga, length=64):
    """Generate one melody sequence as a list of (pitch, duration) tokens."""
    sequence = []
    scale = raga['notes']

    # Start on Sa or with a pakad phrase
    if random.random() < 0.4:
        pakad = random.choice(raga['pakad'])
        for p in pakad:
            sequence.append((p, random.choices(DURATIONS, DUR_WEIGHTS)[0]))
        current = pakad[-1]
    else:
        current = random.choice(raga['samvadi'])
        sequence.append((current, random.choices(DURATIONS, DUR_WEIGHTS)[0]))

    direction  = random.choice(['up', 'down'])
    dir_steps  = 0

    while len(sequence) < length:
        # Inject gamaka ornament
        if random.random() < 0.08:
            for p in random.choice(raga['gamaka']):
                if p in scale:
                    sequence.append((p, 0.25))
            current = sequence[-1][0]
            continue
        # Inject pakad phrase
        if random.random() < 0.06:
            pakad = random.choice(raga['pakad'])
            for p in pakad:
                sequence.append((p, random.choices(DURATIONS, DUR_WEIGHTS)[0]))
            current = pakad[-1]
            continue
        # Flip direction
        dir_steps += 1
        if dir_steps > random.randint(4, 10):
            direction = 'down' if direction == 'up' else 'up'
            dir_steps = 0
        # Sample next note
        weights = get_note_weights(current, raga, direction)
        total   = sum(weights)
        probs   = [w / total for w in weights]
        current = random.choices(scale, weights=probs)[0]
        sequence.append((current, random.choices(DURATIONS, DUR_WEIGHTS)[0]))

    return sequence[:length]


NUM_SEQUENCES = 5000
SEQ_LENGTH    = 64

print(f'Generating {NUM_SEQUENCES} sequences...')
all_sequences = [generate_sequence(BHAIRAV, SEQ_LENGTH) for _ in range(NUM_SEQUENCES)]
print(f'Done. Total tokens: {NUM_SEQUENCES * SEQ_LENGTH:,}')

print('\nSample (first 8 tokens):')
for pitch, dur in all_sequences[0][:8]:
    print(f'  MIDI {pitch:3d} ({note.Note(pitch).nameWithOctave:5s})  dur={dur}')

## 4. Tokenization

Each `(pitch, duration)` pair maps to a single integer token. This keeps the vocabulary compact so the LSTM learns efficiently.

In [ ]:
all_tokens  = [tok for seq in all_sequences for tok in seq]
vocab       = sorted(set(all_tokens))
token2idx   = {t: i for i, t in enumerate(vocab)}
idx2token   = {i: t for t, i in token2idx.items()}
VOCAB_SIZE  = len(vocab)

indexed_sequences = [[token2idx[t] for t in seq] for seq in all_sequences]

print(f'Vocabulary size: {VOCAB_SIZE} unique (pitch, duration) pairs')

freq = Counter(all_tokens)
print('\nTop 5 most common tokens:')
for tok, count in freq.most_common(5):
    p, d = tok
    print(f'  {note.Note(p).nameWithOctave:5s}  dur={d}  →  {count}x  ({100*count/len(all_tokens):.1f}%)')

## 5. Dataset — Sliding Window

Each training sample: window of 32 tokens as input → next token as target. Standard next-token prediction (same framing as language modeling).

In [ ]:
WINDOW_SIZE = 32

class RagaDataset(Dataset):
    def __init__(self, sequences, window_size):
        self.samples = []
        for seq in sequences:
            for i in range(len(seq) - window_size):
                self.samples.append((seq[i:i+window_size], seq[i+window_size]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)


split      = int(0.9 * len(indexed_sequences))
train_ds   = RagaDataset(indexed_sequences[:split], WINDOW_SIZE)
val_ds     = RagaDataset(indexed_sequences[split:], WINDOW_SIZE)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, num_workers=2)

print(f'Train samples: {len(train_ds):,}')
print(f'Val samples:   {len(val_ds):,}')
print(f'Batches/epoch: {len(train_loader):,}')

## 6. LSTM Model

- **Embedding**: token index → 64-dim vector
- **2-layer LSTM**: 128 hidden units, dropout=0.5
- **Linear head**: projects to vocabulary size

In [ ]:
class RagaLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, num_layers=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm      = nn.LSTM(embed_dim, hidden_dim, num_layers,
                                 dropout=dropout, batch_first=True)
        self.drop      = nn.Dropout(dropout)
        self.fc        = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb            = self.drop(self.embedding(x))   # (B, T, E)
        out, hidden    = self.lstm(emb, hidden)         # (B, T, H)
        logits         = self.fc(self.drop(out[:,-1,:])) # last step → (B, V)
        return logits, hidden


model = RagaLSTM(VOCAB_SIZE).to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(model)

## 7. Training

Cross-entropy loss, Adam optimizer, ReduceLROnPlateau scheduler, gradient clipping, early stopping.

In [ ]:
EPOCHS    = 50
LR        = 1e-3
PATIENCE  = 7

criterion  = nn.CrossEntropyLoss()
optimizer  = torch.optim.Adam(model.parameters(), lr=LR)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5, verbose=True)

train_losses, val_losses = [], []
best_val_loss = float('inf')
no_improve    = 0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────
    model.train()
    total = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        logits, _ = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    avg_train = total / len(train_loader)

    # ── Validate ───────────────────────────────────────────
    model.eval()
    total = 0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits, _ = model(xb)
            total += criterion(logits, yb).item()
    avg_val = total / len(val_loader)

    scheduler.step(avg_val)
    train_losses.append(avg_train)
    val_losses.append(avg_val)

    # ── Checkpoint & early stopping ────────────────────────
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        no_improve    = 0
        torch.save(model.state_dict(), 'best_model.pt')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d} | Train: {avg_train:.4f} | Val: {avg_val:.4f} | LR: {optimizer.param_groups[0]["lr"]:.6f}')

print(f'\nBest val loss: {best_val_loss:.4f}')

## 8. Training Curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(train_losses, label='Train', color='steelblue')
plt.plot(val_losses,   label='Val',   color='coral')
plt.xlabel('Epoch'); plt.ylabel('Cross-Entropy Loss')
plt.title('Raga Bhairav LSTM — Training Curves')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 9. Melody Generation

Autoregressive sampling with **temperature scaling**:
- `0.5` — conservative, stays close to learned grammar
- `1.0` — balanced
- `1.5` — more adventurous

Seeded with Bhairav's classic opening pakad phrase.

In [ ]:
def generate_melody(model, seed_tokens, length=96, temperature=1.0):
    model.eval()
    generated = list(seed_tokens)
    with torch.no_grad():
        for _ in range(length):
            context = generated[-WINDOW_SIZE:]
            x       = torch.tensor([context], dtype=torch.long).to(DEVICE)
            logits, _ = model(x)
            probs   = torch.softmax(logits[0] / temperature, dim=-1)
            generated.append(torch.multinomial(probs, 1).item())
    return [idx2token[i] for i in generated[len(seed_tokens):]]


model.load_state_dict(torch.load('best_model.pt', map_location=DEVICE))

# Seed: classic Bhairav opening — Re Sa Dha Pa Ma
seed_phrase = BHAIRAV['pakad'][0]
seed_tokens = []
for p in seed_phrase:
    key = (p, 0.5)
    if key in token2idx:
        seed_tokens.append(token2idx[key])
    else:
        # fallback: nearest token
        seed_tokens.append(token2idx[min(token2idx, key=lambda t: abs(t[0]-p))])

melodies = {}
for temp in [0.5, 1.0, 1.5]:
    mel = generate_melody(model, seed_tokens, length=96, temperature=temp)
    melodies[temp] = mel
    print(f'Temp {temp}: {len(mel)} tokens | first 4: {[(note.Note(p).nameWithOctave, d) for p,d in mel[:4]]}')

## 10. Analysis — Did the Model Learn Bhairav?

We verify the model internalized the grammar by checking scale adherence, vadi emphasis, and interval distribution.

In [ ]:
BHAIRAV_PCS  = {p % 12 for p in BHAIRAV['notes']}  # {0,1,4,5,7,8,11}
NOTE_NAMES   = {0:'Sa(C)', 1:'Re(Db)', 4:'Ga(E)', 5:'Ma(F)',
                7:'Pa(G)', 8:'Dha(Ab)', 11:'Ni(B)'}

for temp in [0.5, 1.0, 1.5]:
    mel     = melodies[temp]
    pitches = [p for p,d in mel]
    pcs     = [p % 12 for p in pitches]

    in_scale  = sum(1 for pc in pcs if pc in BHAIRAV_PCS)
    intervals = [abs(pitches[i+1]-pitches[i]) for i in range(len(pitches)-1)]
    stepwise  = sum(1 for iv in intervals if iv <= 2)

    print(f'\n── Temperature {temp} ──────────────────────────────')
    print(f'Scale adherence : {in_scale}/{len(pitches)} = {100*in_scale/len(pitches):.1f}%')
    print(f'Stepwise motion : {100*stepwise/len(intervals):.1f}%')

    pc_freq = Counter(pcs)
    print('Note distribution:')
    for pc, cnt in pc_freq.most_common():
        name = NOTE_NAMES.get(pc, f'pc={pc}')
        bar  = '█' * int(30 * cnt / len(pitches))
        print(f'  {name:12s} {bar} {cnt}')

## 11. Multi-Track Performance Assembly

We combine three tracks into a full Bhairav performance:

| Track | Instrument | Role |
|-------|-----------|------|
| 1 | Sitar | LSTM-generated melody |
| 2 | Shamisen (Tanpura) | Sa–Pa–Sa–Sa drone loop |
| 3 | Percussion (Tabla) | Teentaal 16-beat cycle |

### Teentaal Structure
16 beats divided into 4 vibhags: **Dha Dhin Dhin Dha** | **Dha Dhin Dhin Dha** | **Na Tin Tin Na** | **Dha Dhin Dhin Dha**

The 3rd vibhag (Na Tin Tin Na) is the *khali* (empty) section — no baya (left hand) stroke, lighter texture.

In [ ]:
# ── Teentaal pattern ──────────────────────────────────────────────────────────
# Each entry: (syllable, baya_midi, tabla_midi, duration_qtr, baya_vel, tabla_vel)
# GM percussion: 36=Bass Drum, 37=Side Stick, 38=Snare, 42=Closed Hi-Hat
TEENTAAL = [
    # Vibhag 1
    ('Dha',  36, 38, 0.5, 95, 90),  # Sam — strongest beat
    ('Dhin', 36, 38, 0.5, 70, 80),
    ('Dhin', 36, 38, 0.5, 65, 75),
    ('Dha',  36, 38, 0.5, 80, 80),
    # Vibhag 2
    ('Dha',  36, 38, 0.5, 85, 80),
    ('Dhin', 36, 38, 0.5, 65, 70),
    ('Dhin', 36, 38, 0.5, 60, 70),
    ('Dha',  36, 38, 0.5, 75, 75),
    # Vibhag 3 — Khali (no baya)
    ('Na',   0,  37, 0.5,  0, 70),
    ('Tin',  0,  42, 0.5,  0, 65),
    ('Tin',  0,  42, 0.5,  0, 60),
    ('Na',   0,  37, 0.5,  0, 65),
    # Vibhag 4
    ('Dha',  36, 38, 0.5, 85, 80),
    ('Dhin', 36, 38, 0.5, 65, 75),
    ('Dhin', 36, 38, 0.5, 60, 70),
    ('Dha',  36, 38, 0.5, 80, 80),
]

# ── Tanpura drone: Sa–Pa–Sa–Sa (low octave, very soft) ────────────────────────
TANPURA = [
    (48, 2.0, 40),   # low Sa  (C3)
    (55, 2.0, 35),   # Pa      (G3)
    (48, 2.0, 38),   # low Sa  (C3)
    (60, 2.0, 42),   # mid Sa  (C4)
]


def build_tabla_part(total_duration):
    part = stream.Part()
    part.id = 'Tabla'
    part.append(instrument.UnpitchedPercussion())
    elapsed, idx = 0.0, 0
    while elapsed < total_duration:
        _, baya_midi, tabla_midi, dur, baya_vel, tabla_vel = TEENTAAL[idx % len(TEENTAAL)]
        if baya_midi > 0:
            n = note.Note(); n.pitch.midi = baya_midi
            n.quarterLength = dur * 0.4; n.volume.velocity = baya_vel
            part.append(n)
        n2 = note.Note(); n2.pitch.midi = tabla_midi
        n2.quarterLength = dur; n2.volume.velocity = tabla_vel
        part.append(n2)
        elapsed += dur; idx += 1
    return part


def build_tanpura_part(total_duration):
    part = stream.Part()
    part.id = 'Tanpura'
    part.append(instrument.Shamisen())  # closest GM timbre to tanpura
    elapsed, idx = 0.0, 0
    while elapsed < total_duration:
        midi_p, dur, vel = TANPURA[idx % len(TANPURA)]
        n = note.Note(midi_p); n.quarterLength = dur; n.volume.velocity = vel
        part.append(n)
        elapsed += dur; idx += 1
    return part


def build_full_performance(melody_tokens, filename, bpm=55):
    """
    Assemble 3-track Bhairav performance and write to MIDI.
    Track 1: Sitar melody (LSTM-generated)
    Track 2: Tanpura drone (Sa-Pa-Sa-Sa loop)
    Track 3: Tabla percussion (Teentaal 16-beat cycle)
    """
    total_dur = sum(d for _, d in melody_tokens)
    s = stream.Score()

    # Track 1: Melody
    mel_part = stream.Part(); mel_part.id = 'Melody'
    mel_part.append(instrument.Sitar())
    mel_part.append(tempo.MetronomeMark(number=bpm))
    for pitch_midi, dur in melody_tokens:
        n = note.Note(pitch_midi); n.quarterLength = dur; n.volume.velocity = 85
        mel_part.append(n)

    # Track 2: Tanpura
    tanpura_part = build_tanpura_part(total_dur)

    # Track 3: Tabla
    tabla_part = build_tabla_part(total_dur)

    s.append(mel_part)
    s.append(tanpura_part)
    s.append(tabla_part)
    s.write('midi', fp=filename)

    print(f'Saved: {filename}')
    print(f'  Duration: {total_dur:.0f} quarter notes @ {bpm} BPM ≈ {total_dur/bpm*60:.0f}s')
    print(f'  Tracks: Sitar | Tanpura | Tabla (Teentaal)')
    return filename


print('Multi-track functions defined.')
print(f'Teentaal cycle: {len(TEENTAAL)} beats, {sum(b[3] for b in TEENTAAL)} quarter notes/cycle')

## 12. Export MIDI → MP3

In [ ]:
SOUNDFONT = '/usr/share/sounds/sf2/FluidR3_GM.sf2'

def midi_to_mp3(midi_path, mp3_path, soundfont=SOUNDFONT):
    wav = midi_path.replace('.mid', '.wav')
    subprocess.run(['fluidsynth', '-ni', soundfont, midi_path, '-F', wav, '-r', '44100'],
                   check=True, capture_output=True)
    subprocess.run(['ffmpeg', '-y', '-i', wav, '-codec:a', 'libmp3lame', '-qscale:a', '2', mp3_path],
                   check=True, capture_output=True)
    os.remove(wav)
    print(f'MP3 saved: {mp3_path}')


# Generate full performances at all three temperatures
for temp, mel in melodies.items():
    mid_path = f'bhairav_full_temp{temp}.mid'
    mp3_path = f'bhairav_full_temp{temp}.mp3'
    build_full_performance(mel, mid_path, bpm=55)
    midi_to_mp3(mid_path, mp3_path)
    print()

## 13. Listen

In [ ]:
from IPython.display import Audio, display

for temp in [0.5, 1.0, 1.5]:
    print(f'\n🎵 Temperature = {temp}')
    display(Audio(f'bhairav_full_temp{temp}.mp3'))

## 14. Save Checkpoint

In [ ]:
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab':      vocab,
    'token2idx':  token2idx,
    'idx2token':  idx2token,
    'window_size': WINDOW_SIZE,
    'raga':       BHAIRAV['name'],
}, 'raga_bhairav_checkpoint.pt')

files = [
    'raga_bhairav_checkpoint.pt', 'training_curves.png',
    'bhairav_full_temp0.5.mid',  'bhairav_full_temp1.0.mid',  'bhairav_full_temp1.5.mid',
    'bhairav_full_temp0.5.mp3',  'bhairav_full_temp1.0.mp3',  'bhairav_full_temp1.5.mp3',
]
print('Output files:')
for f in files:
    status = f'✅ {os.path.getsize(f):>10,} bytes' if os.path.exists(f) else '❌ missing'
    print(f'  {status}  {f}')